# Pandas & NumPy

## Introduction

Welcome! This notebook gives you a fast, hands-on tour of the two most important Python libraries for data science: **Pandas** and **NumPy**.

You do not need any prior experience with these libraries. All you need is basic Python, variables, lists, and loops.

By the end of this session, you will have run real code against real datasets and seen exactly what these tools can do.

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Import and use `pandas` and `numpy` correctly

2. Load a CSV file and explore its contents

3. Select, filter, and sort data from a DataFrame

4. Group data and compute summary statistics

5. Create new columns and handle missing values

6. Create basic plots directly from a DataFrame

7. Create and manipulate NumPy arrays

8. Work with dates and combine multiple DataFrames

---

## Part 1: Key Imports

Every data science script starts with **imports**, telling Python which external libraries to load.

- **`pandas`**: for tables of data (rows + columns), called DataFrames

- **`numpy`**: for fast numerical arrays and maths

- **`matplotlib.pyplot`**: for creating plots and charts

The `as` keyword gives a library a **short alias** so we don't have to type the full name every time.

> **Convention:** Always use `pd`, `np`, and `plt` as aliases. Every data scientist in the world uses these same names.

> **Discussion:** Why do you think we use short aliases like `pd` instead of typing `pandas` every time?

In [ ]:
# Import the three core libraries for data science
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Confirm versions, useful when debugging or sharing code
print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

---
## Part 2: Pandas, Loading & Exploring Data

### What is a DataFrame?

Think of a **DataFrame** as a spreadsheet inside Python, a table made of **rows** and **columns**. Each column has a name, and every row is one observation.

We load data from a CSV file using `pd.read_csv()`. That's it, one line.

> **Discussion:** What kinds of data do you work with in your day-to-day life that might look like a table?

In [ ]:
# Load the Iris dataset, a classic dataset about flower measurements
df = pd.read_csv(
    "data/iris.csv"
)  # there are also .read_excel() for Excel files, and .read_json() for JSON files

# Show the first 5 rows to get a feel for the data
df.head()

In [ ]:
# --- Shape and column names ---
print("Rows, Columns:", df.shape)
print("Column names :", df.columns.tolist())

In [ ]:
# --- Quick data health check ---
# .info() shows column types and whether any values are missing
df.info()

In [ ]:
# --- Summary statistics for all numeric columns ---
# .describe() gives count, mean, std, min, max, and quartiles
df.describe().round(2)

### Exploring Categorical Columns

The `species` column is **categorical**: it holds text labels, not numbers.

- `.unique()`: shows the distinct values

- `.value_counts()`: shows how many rows belong to each category

> **Discussion:** What does it tell you if one category has far more rows than others?

In [ ]:
# --- Explore the categorical 'species' column ---

# How many unique species are there?
print("Unique species:", df["species"].unique())

# --- How many rows belong to each species? ---
print("\nCounts per species:")
print(df["species"].value_counts())

---

## Part 3: Selecting & Filtering Data

### Selecting Columns

Use square brackets `[]` to pick columns, just like looking up a **key** in a dictionary.

- `df['column_name']` → returns one column (this is a **Series**)

- `df[['col_1', 'col_2']]` → returns multiple columns (this is a **DataFrame**)

### Selecting Rows by Position: `.iloc[]`

**`iloc`** stands for *integer location*. Use it when you want rows by their position number (0, 1, 2 …)

### Selecting Rows by Label: `.loc[]`

**`loc`** selects by **label**, usually the index value or a condition.

> **Discussion:** When would you use `.iloc[]` vs `.loc[]`? Can you think of a real-world example for each?

In [ ]:
# --- Select a single column ---
sepal_lengths = df["sepal_length"]
print(type(sepal_lengths))
print(sepal_lengths.head())

In [ ]:
# --- Select multiple columns ---
measurements = df[["sepal_length", "petal_length", "species"]]
print(type(measurements))
measurements.head()

In [ ]:
# --- iloc: select by position (row index number) ---
# Get rows index 0 to 4, all columns
df.iloc[0:5]  # this is the same as calling df.head()

In [ ]:
# --- loc: select by condition (boolean mask) ---
# Get all rows where species is 'setosa'
setosa_df = df.loc[df["species"] == "Iris-setosa"]
print("Setosa rows:", len(setosa_df))
setosa_df.head()

### Filtering with Multiple Conditions

Combine conditions with:

- `&`: AND (both must be true)

- `|`: OR (either can be true)

> **Important:** Each condition must be wrapped in `()` when combining.

> **Discussion:** How would you find all flowers with a sepal length above the average AND a petal length above 4?

In [ ]:
# --- Filter: setosa species with sepal length > 5 ---
filtered = df[(df["species"] == "Iris-setosa") & (df["sepal_length"] > 5)]
print("Rows matching filter =", len(filtered))
filtered.head()

In [ ]:
# --- Filter using .query(), a more readable alternative ---
# query() lets you write the condition as a plain-English string
result = df.query("species == 'Iris-setosa' and sepal_length > 5")
print("Rows matching query =", len(result))

---
## Part 4: Sorting & Grouping

### Sorting

`.sort_values('column')` arranges rows in ascending order by that column.

Add `ascending=False` to flip to descending order.

### Groupby: Split → Apply → Combine

This is one of the most powerful Pandas patterns:

1. **Split** the data into groups (e.g., by species)

2. **Apply** an operation to each group (e.g., calculate the mean)

3. **Combine** the results back into one table'

> **Discussion:** Can you think of a business question where you'd want to compare averages across groups? (e.g., sales by region, scores by class)

In [ ]:
# --- Sort by petal_length, largest first ---
df.sort_values("petal_length", ascending=False).head(5)

In [ ]:
# --- Groupby: average measurements per species ---
# Step 1: group rows by species
# Step 2: calculate the mean of each numeric column
species_means = df.groupby("species").mean()
species_means

In [ ]:
# --- Groupby with a specific column and multiple aggregations ---
# .agg() lets you apply several functions at once
petal_stats = df.groupby("species")["petal_length"].agg(["mean", "min", "max"])
petal_stats

---
## Part 5: Column Operations & Handling Missing Data

### Creating New Columns

You can create a new column using arithmetic on existing columns. Pandas applies the operation **row by row** automatically, no loop needed.

### Renaming & Dropping Columns

- `.rename(columns={'old': 'new'})`: rename one or more columns

- `.drop(columns=['col'])`: remove a column

### Missing Values

Real-world data is messy. Missing values show up as `NaN` (Not a Number).

- `.isnull().sum()`: count missing values per column

- `.fillna(value)`: replace NaN with a given value

- `.dropna()`: remove rows that contain any NaN

> **Discussion:** If a column has 30% missing values, would you fill them or drop the rows? What factors would influence your decision?

In [ ]:
# --- Create a new column: petal area (length × width) ---
df["petal_area"] = df["petal_length"] * df["petal_width"]
df[["species", "petal_length", "petal_width", "petal_area"]].head()

In [ ]:
# --- Check for missing values in the dataset ---
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# --- Demonstrate fillna and dropna with an example ---

# Create a small DataFrame with some missing values
sample_data = pd.DataFrame(
    {"name": ["Alice", "Bob", "Carol", "Dan"], "score": [85, None, 90, None]}
)

print("Original:")
print(type(sample_data))
print(sample_data)

In [ ]:
# --- Fill missing scores with the column mean ---
print("\nAfter fillna with mean:")
print(sample_data["score"].fillna(sample_data["score"].mean()))

---
## Part 6: Quick Visualisation

Pandas has built-in plotting via **Matplotlib**. You call `.plot()` on a DataFrame or Series and it produces a chart.

Common chart types:

| `kind`       | Use case |
|---------------|----------|
| `'hist'`      | Distribution of one variable |
| `'scatter'`   | Relationship between two variables |
| `'bar'`       | Comparing categories |
| `'box'`       | Spread and outliers per group |


**Discussion:** Which chart type would you use to check if a column has outliers? Which would you use to spot trends over time?

In [ ]:
# --- Histogram: distribution of sepal length ---
df["sepal_length"].plot(
    kind="hist", title="Sepal Length Distribution"
)  # default bins = 10 -> 20 is often better for larger datasets
plt.xlabel("Sepal Length (cm)")
plt.show()

In [ ]:
# --- Scatter plot: petal length vs petal width ---
# alpha controls transparency so overlapping points are visible
df.plot(
    kind="scatter",
    x="petal_length",
    y="petal_width",
    alpha=0.5,
    title="Petal Length vs Width",
)
plt.show()

In [ ]:
# --- Bar chart: mean petal length per species ---
# Chain groupby → mean → plot for a one-liner insight
df.groupby("species")["petal_length"].mean().plot(
    kind="bar", title="Average Petal Length by Species", color="steelblue"
)
plt.ylabel("Mean Petal Length (cm)")
plt.xticks(rotation=45)
plt.show()

---
## Part 7: NumPy, Fast Numerical Arrays

### Why NumPy?

A Python list is flexible but slow for maths. A **NumPy array** stores all values as the same type in contiguous memory; this makes it **10–100x faster** for numerical operations.

NumPy is also the foundation that Pandas is built on. Every column in a DataFrame is a NumPy array underneath.

### Key Concepts

- **Array:** a grid of numbers, all the same type

- **Vectorisation:** applying an operation to every element at once, no loop needed

- **Broadcasting:** applying an operation between arrays of different (but compatible) shapes

> **Discussion:** Can you think of a scenario where speed matters when processing numbers? (e.g., images, simulations, real-time data)

In [ ]:
# --- Speed demo: NumPy vs Python list ---
python_list = list(range(1_000_000))
numpy_array = np.arange(1_000_000)

# %timeit measures how long a line takes to run (run multiple times for accuracy)
%timeit sum(python_list)
%timeit np.sum(numpy_array)

In [ ]:
# --- Creating arrays ---

# From a Python list
arr = np.array([10, 20, 30, 40, 50, 60])

# --- A range of numbers ---
range_arr = np.arange(0, 10, 2)  # (start, stop, step): [0, 2, 4, 6, 8]

lin_arr = np.linspace(
    0, 1, 5
)  # 5 numbers evenly spaced between 0 and 1: [0. , 0.25, 0.5 , 0.75, 1.]

log_arr = np.logspace(
    0, 3, 4
)  # 4 numbers evenly spaced on a log scale from 10^0 to 10^3: [   1.   10.  100. 1000.]

zero_arr = np.zeros((3, 3))  # 3 rows, 3 columns of zeros

# --- A 2D array (like a matrix) ---
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print("1D array     :", arr)
print("Range arr    :", range_arr)
print("Linspace arr :", lin_arr)
print("Logspace arr :", log_arr)
print("Matrix shape :", matrix.shape)
print(matrix)
print("---")
print(zero_arr)

In [ ]:
# --- Vectorised arithmetic: no loop needed ---
prices = np.array([100, 200, 150, 300, 250])

# Apply a 10% discount to every price at once
discounted = prices * 0.9
print("Original :", prices)
print("Discounted:", discounted)

In [ ]:
# --- Indexing and slicing a 2D array ---
# Syntax: matrix[row, column], use : to mean 'all'
print("-matrix-")
print(matrix)
print("Element at row 0, col 2 :", matrix[0, 2])  # → 3
print("Entire first row        :", matrix[0, :])  # → [1 2 3]
print("Entire second column    :", matrix[:, 1])  # → [2 5 8]

In [ ]:
# --- Reshaping arrays ---
# Change a 1D array of 6 elements into a 2D array with 2 rows and 3 columns
print(arr)
reshaped = arr.reshape(2, 3)
reshaped_2 = arr.reshape(3, 2)  # Reshape to 3 rows and 2 columns
print("Reshaped array:")
print(reshaped)
print(reshaped_2)

In [ ]:
# --- Aggregations with the axis parameter ---
# axis=0 collapses ROWS → gives one result per COLUMN
# axis=1 collapses COLUMNS → gives one result per ROW

print("Column sums(axis=0):", matrix.sum(axis=0))  # [12 15 18]
print("Row sums   (axis=1):", matrix.sum(axis=1))  # [6 15 24]
print("Overall mean       :", matrix.mean())  # 5.0

In [ ]:
# --- np.where: find elements matching a condition ---
# Returns the indices where the condition is True
scores = np.array([45, 72, 88, 35, 91, 60])
passing_indices = np.where(scores >= 60)

print("Passing indices:", passing_indices)
print("Passing scores :", scores[passing_indices])

---
## Part 8: Datetime Handling

Dates and times are stored as plain text in CSV files. Pandas can convert them to proper **datetime objects**, which unlocks powerful time-based operations.

### Workflow

1. Load the CSV: dates arrive as strings

2. Convert with `pd.to_datetime()`

3. Extract components using the **`.dt` accessor** (`.dt.year`, `.dt.month`, `.dt.day`, `.dt.weekday`)

4. Use date arithmetic (add/subtract days, calculate durations)

**Discussion:** What new analysis questions become possible once you can extract the month or day-of-week from a date column?

In [ ]:
# --- Load the Seattle weather dataset (has a date column) ---
weather = pd.read_csv("data/seattle-weather.csv")
print("date column type before conversion:", weather["date"].dtype)
weather.head(3)

In [ ]:
# --- Convert the 'date' column from string to datetime ---
weather["date"] = pd.to_datetime(weather["date"])
print("date column type after conversion:", weather["date"].dtype)
weather.head(3)

In [ ]:
# --- Extract date components using the .dt accessor ---
weather["year"] = weather["date"].dt.year

weather["month"] = weather["date"].dt.month  # 1 = January, obviously.

weather["weekday"] = weather["date"].dt.weekday  # 0 = Monday, 6 = Sunday

weather[["date", "year", "month", "weekday"]].head(5)

In [ ]:
# --- Practical insight: average temperature by month ---

max_temp = weather.groupby("month")["temp_max"].mean()
max_temp.plot(kind="bar", title="Average Max Temp by Month (Seattle)", color="coral")

"""min_temp = weather.groupby('month')['temp_min'].mean()
min_temp.plot(
    kind='bar', 
    title='Average Max/Min Temp by Month (Seattle)',
    color='lightblue')"""

plt.xlabel("Month")
plt.ylabel("Avg Temp (°C)")
# plt.legend(['Max Temp', 'Min Temp'])
plt.xticks(rotation=0)
plt.show()

---
## Part 9: Combining DataFrames

Real data is rarely in one table. You will often need to combine datasets.

### Three main methods

| Method | What it does |
|---|---|
| `pd.concat([df1, df2], axis=0)` | Stack rows on top of each other |
| `pd.concat([df1, df2], axis=1)` | Place columns side by side |
| `pd.merge(left, right, on='key', how='inner')` | Join on a shared column (like SQL) |

### Join types for `pd.merge()`

- **`inner`**: only rows that match in **both** tables
- **`left`**: all rows from the left, matching rows from the right
- **`outer`**: all rows from both tables (fills NaN where no match)

**Discussion:** Imagine you have a customer table and an orders table. Which join type would you use to find all customers, even those who haven't ordered yet?

In [ ]:
# --- Demonstrate pd.concat: stack two small DataFrames vertically ---
batch_1 = pd.DataFrame({"name": ["Alice", "Bob"], "score": [85, 92]})
batch_2 = pd.DataFrame({"name": ["Carol", "Dan"], "score": [78, 88]})
print("Batch 1:")
print(batch_1)
print("\nBatch 2:")
print(batch_2)

In [ ]:
# Combine into one table, sort the scores and reset the index
all_students = (
    pd.concat([batch_1, batch_2], axis=0)
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)
print("Combined:")
print(all_students)

In [ ]:
# --- Demonstrate pd.merge: join two tables on a shared column ---
students = pd.DataFrame({"id": [1, 2, 3, 4], "name": ["Alice", "Bob", "Carol", "Dan"]})
grades = pd.DataFrame({"id": [1, 2, 3], "grade": ["A", "B", "A"]})

# --- inner join: only students who have a grade ---
inner_result = pd.merge(students, grades, on="id", how="inner")
print("Inner join (only matched rows):")
print(inner_result)

# --- left join: all students, NaN if no grade ---
left_result = pd.merge(students, grades, on="id", how="left")
print("\nLeft join (all students):")
print(left_result)

In [ ]:
# --- Real example: load and merge red + white wine datasets ---
red_wine = pd.read_csv("data/winequality-red.csv", sep=";")
white_wine = pd.read_csv("data/winequality-white.csv", sep=";")
red_wine.head(2)

In [ ]:
white_wine.head(2)

In [ ]:
# Add a label column before combining
red_wine["type"] = "red"
white_wine["type"] = "white"

# Stack both datasets into one
wine = pd.concat([red_wine, white_wine], axis=0).reset_index(drop=True)

print("Combined wine dataset shape:", wine.shape)
print(wine["type"].value_counts())
wine.head()

In [ ]:
# --- Compare average quality between red and white wine ---
wine.groupby("type")["quality"].mean().plot(
    kind="bar", title="Average Wine Quality by Type", color=["darkred", "gold"]
)
plt.ylabel("Average Quality Score")
plt.xticks(rotation=0)
plt.show()

---

## Key Takeaways

Here is what you covered in this session:

**Pandas**

- Load data with `pd.read_csv()` and explore with `.head()`, `.info()`, `.describe()`

- Select columns with `[]`, rows by position with `.iloc[]`, rows by condition with `.loc[]`

- Filter with boolean masks, wrap each condition in `()`

- Aggregate with `.groupby()` → one of the most used patterns in data science

- Create new columns with direct arithmetic on existing columns

- Handle missing values with `.fillna()` and `.dropna()`

- Plot directly with `.plot(kind='hist'/'bar'/'scatter'/'box')`

**NumPy**

- Arrays are faster than lists for numerical work

- No loops needed: operations apply element-wise automatically

- Use `axis=0` for column-wise, `axis=1` for row-wise aggregations

- `np.where()` finds indices matching a condition

**Datetime**

- Convert string columns with `pd.to_datetime()`

- Extract year, month, day, weekday using the `.dt` accessor

**Combining Data**

- Stack rows with `pd.concat(axis=0)`

- Join on a key column with `pd.merge()`: choose join type carefully